<a href="https://colab.research.google.com/github/sudikshyapant/unet/blob/main/unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Encoder

In [1]:
import tensorflow as tf

def encoder_block(num_filter, input):

  x = tf.keras.layers.Conv2D(num_filter, 3, padding = "valid")(input)
  x = tf.keras.layers.Activation('relu')(x)

  x = tf.keras.layers.Conv2D(num_filter, 3, padding = "valid")(x)
  x = tf.keras.layers.Activation('relu')(x)

  x = tf.keras.layers.MaxPool2D(pool_size=(2,2), strides= 2)(x)

  return x

## Decoder

In [2]:
from numpy import interp
import tensorflow as tf

def decoder_block(num_filter, input, skip_features):

  x = tf.keras.layers.Conv2DTranspose(num_filter, 2, strides=(2,2), padding="valid")(input)

  skip_features = tf.keras.layers.Resizing(x.shape[1], x.shape[2], interpolation= 'bilinear')(skip_features)

  x =  tf.keras.layers.Concatenate()([x, skip_features])

  x = tf.keras.layers.Conv2D(num_filter, 3, padding = "valid")(input)
  x = tf.keras.layers.Activation('relu')(x)

  x = tf.keras.layers.Conv2D(num_filter, 3, padding = "valid")(x)
  x = tf.keras.layers.Activation('relu')(x)

  return x

## The U-Net model

In [4]:
def unet_model(input_shape= (256,256,3), num_class=1):
  input = tf.keras.layers.Input(shape=input_shape)

# Contracting part

  s1 = encoder_block(64, input)
  s2 = encoder_block(128, s1)
  s3 = encoder_block(256, s2)
  s4 = encoder_block(512, s3)

# Bottleneck
  b1 = tf.keras.layers.Conv2D(1024, 3, padding = 'valid')(s3)
  b1 = tf.keras.layers.Activation('relu')
  b1 = tf.keras.layers.Conv2D(1024, 3, padding = 'valid')(s3)
  b1 = tf.keras.layers.Activation('relu')

# Decoder
  d1 = decoder_block(512, b1, s4)
  d2 = decoder_block(256, d1, s3)
  d3 = decoder_block(128, d2, s2)
  d4 = decoder_block(64, d3, s1)